# Fixed EF point-scale prediction

Calculate the primary fixed exponential-filter baseline directly from prepared ASCAT or SMAP SSM. The characteristic time is fixed at 15 days for both products; in-situ RZSM is retained in outputs for downstream validation but is never used to generate the prediction.

In [ ]:
from config import configure_runtime
configure_runtime()


In [ ]:
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from config import EF_WORKERS, results_FP
from EF.integrity import atomic_write_csv, clean_matching_files
from EF.prediction import (
    SUMMARY_COLUMNS, canonical_pixel_files, prediction_run_status,
    process_station_file,
)


## Settings and paths

In [ ]:
PRODUCTS = ("ASCAT", "SMAP")
AREAS = ("Train", "Test", "Excluded")
FIXED_T_DAYS = 15.0
SPIN_UP_DAYS = 365.0
STUDY_START = "2015-04-01"
STUDY_END = "2023-12-31"
EXPECTED_DAYS = 3197
TARGET_VARIABLE = "in-situ_RZSM"
GRID_RESOLUTION = 0.1

RESULTS = Path(results_FP)
ISMN_ROOT = RESULTS / "ISMN"
COHORT_ROOT = ISMN_ROOT / "Station_TCA_Screening"
OUTPUT_ROOT = RESULTS / "EF" / "Fixed_EF"

GRID_ROWS = int(round(180.0 / GRID_RESOLUTION))
GRID_COLUMNS = int(round(360.0 / GRID_RESOLUTION))
LATITUDE_AXIS = np.linspace(90.0 - GRID_RESOLUTION / 2.0, -90.0 + GRID_RESOLUTION / 2.0, GRID_ROWS)
LONGITUDE_AXIS = np.linspace(-180.0 + GRID_RESOLUTION / 2.0, 180.0 - GRID_RESOLUTION / 2.0, GRID_COLUMNS)
print(f"Output: {OUTPUT_ROOT}")


## Generate Fixed EF predictions

In [ ]:
run_summaries = []
for area in AREAS:
    cohort_csv = COHORT_ROOT / f"{area}_pixel_list.csv"
    for product in PRODUCTS:
        input_directory = ISMN_ROOT / f"ISMN_{area}" / product
        filenames = canonical_pixel_files(
            cohort_csv, input_directory, STUDY_START, STUDY_END, EXPECTED_DAYS
        )
        output_directory = OUTPUT_ROOT / area / product
        output_directory.mkdir(parents=True, exist_ok=True)
        clean_matching_files(output_directory, "*_prediction.nc")
        clean_matching_files(output_directory, "*_prediction.nc.tmp.*")
        tasks = [
            {
                "filename": filename,
                "input_directory": input_directory,
                "output_directory": output_directory,
                "latitude_axis": LATITUDE_AXIS,
                "longitude_axis": LONGITUDE_AXIS,
                "data_type": product,
                "target_variable": TARGET_VARIABLE,
                "filter_time_days": FIXED_T_DAYS,
                "spin_up_days": SPIN_UP_DAYS,
                "filter_policy": "fixed_T15",
            }
            for filename in filenames
        ]
        workers = min(EF_WORKERS, len(tasks))
        if workers == 1:
            results = [process_station_file(task) for task in tqdm(tasks)]
        else:
            with mp.get_context("spawn").Pool(workers) as pool:
                results = list(tqdm(pool.imap(process_station_file, tasks), total=len(tasks)))
        status = prediction_run_status(filenames, results, output_directory)
        if status["missing_outputs"]:
            raise RuntimeError(f"Missing Fixed EF outputs: {status['missing_outputs'][:10]}")
        summary = pd.DataFrame(results, columns=SUMMARY_COLUMNS).sort_values(["lat_idx", "lon_idx"])
        summary_file = OUTPUT_ROOT / area / f"Fixed_EF_{product}_summary.csv"
        atomic_write_csv(summary, summary_file)
        run_summaries.append({"area": area, "product": product, "T_days": FIXED_T_DAYS, "pixels": len(summary), "summary": summary_file})
display(pd.DataFrame(run_summaries))
